# 90—Publish snapshot to Cloud Storage

### ⚠️ ROI maintainers only. This is not a student notebook.

Students run `c1_01_load_explore.ipynb`, which pulls live from NOAA. That is the right experience:
they see provenance, and the cleaning decisions are the teaching.

But a live-source notebook has one failure mode we cannot accept—if NOAA's service is slow or
down on event day, every data lane in the room stalls at once with no way forward. So we publish
a snapshot, and `data_plane/scripts/load.sh` rebuilds from it.

## The important design decision

**This notebook does not reimplement the transformations.** It downloads the student notebook,
runs it once per city with `CITY` overridden, and exports whatever tables that produces.

That matters. If we maintained a second copy of the cleaning logic, the two would drift, and the
fallback would quietly stop matching what students get. Here the snapshot is *by construction*
exactly the output of the notebook we ship. Change the notebook, re-run this, and the fallback
follows.

## Bucket layout

```
gs://class-demo/a4i-2026/<challenge>/<city>/<table>/*.parquet
```

`a4i-2026` is the generic top level so every challenge shares it. Each challenge gets a folder,
each city a subfolder under that, each table a folder of Parquet parts.

In [ ]:
# --- Configuration ---------------------------------------------------------
BUCKET_ROOT = "gs://class-demo/a4i-2026"
CHALLENGE   = "challenge-1-walkability"
DATASET     = "a4i_climate"
LOCATION    = "US"

# The student notebook, fetched live so this cannot drift from what ships.
# Repo: https://github.com/haggman/A4I2026-challenge-1-walkability
NOTEBOOK_URL = (
    "https://raw.githubusercontent.com/haggman/"
    "A4I2026-challenge-1-walkability/main/notebooks/c1_01_load_explore.ipynb"
)

# Cities to publish. These are the ones surfaced in the student notebook's
# config comment, so the fallback covers whatever most teams will pick.
# Note: two of the four host cities (Toronto, Sunnyvale) have no Heat Watch
# campaign, so we do not organize this list around host cities at all.
CITIES = [
    "Dallas",                       # notebook default
    "New York (Manhattan/Bronx)",
    "New York (Brooklyn)",
    "Los Angeles (Central)",
    "Chicago",
    "Houston",
    "Atlanta",
    "Seattle",
    "San Jose",
]

TABLES = ["heat_observations", "station_daily_temps", "tract_demographics"]

import google.auth
from google.cloud import bigquery

_, PROJECT_ID = google.auth.default()
client = bigquery.Client(project=PROJECT_ID)

print(f"Project : {PROJECT_ID}")
print(f"Target  : {BUCKET_ROOT}/{CHALLENGE}/<city>/")
print(f"Cities  : {', '.join(CITIES)}")

## Fetch the student notebook and extract its code

We skip `%%bigquery` magic cells—they are display-only exploration and cannot be executed as
plain Python. Everything that builds a table is an ordinary code cell.

In [ ]:
import re, requests, nbformat

resp = requests.get(NOTEBOOK_URL, timeout=60)
resp.raise_for_status()
student_nb = nbformat.reads(resp.text, as_version=4)

STUDENT_CODE = [
    c.source for c in student_nb.cells
    if c.cell_type == "code" and not c.source.lstrip().startswith("%%")
]

print(f"Fetched {len(student_nb.cells)} cells, {len(STUDENT_CODE)} executable.")

## Build and export, one city at a time

Each city gets a fresh namespace, is built, and is exported immediately—**one pass, not two.**
An earlier version of this notebook built every city, then looped again to export, which
rebuilt each city a second time and roughly doubled the runtime for no benefit.

**We check the student notebook's validation results before exporting.** Worth being precise
about why: the student notebook *prints* its PASS/FAIL table but deliberately does not raise on a
soft failure—a team shouldn't have their notebook die on a warning. Only the hard guards (zero
rows from GHCN or ACS) stop it.

That means a city could produce a technically-loaded but subtly wrong table and still export.
So this notebook reads `CHECKS` out of the executed namespace and refuses to publish any city
with a failing check. **We would rather publish nothing for a city than publish something
wrong**, because a bad snapshot fails silently at 9am on event day.

In [ ]:
import traceback, time

def slugify(city):
    return re.sub(r"[^a-z0-9]+", "-", city.lower()).strip("-")


def build_city(city):
    '''Execute the student notebook with CITY overridden.'''
    ns = {"__name__": "__main__"}
    for src in STUDENT_CODE:
        src = re.sub(r'^CITY\s*=\s*["\'].*?["\']',
                     f'CITY = "{city}"', src, count=1, flags=re.M)
        exec(compile(src, f"<{slugify(city)}>", "exec"), ns)
    return ns


def export_city(city):
    base = f"{BUCKET_ROOT}/{CHALLENGE}/{slugify(city)}"
    for table in TABLES:
        uri = f"{base}/{table}/*.parquet"
        client.query(f'''
            EXPORT DATA OPTIONS(
              uri = '{uri}',
              format = 'PARQUET',
              overwrite = true
            ) AS
            SELECT * FROM `{PROJECT_ID}.{DATASET}.{table}`
        ''').result()
        print(f"   exported {table}")
    return base


published, failed = {}, {}
run_start = time.time()

for city in CITIES:
    print("=" * 60)
    print(f"{city}")
    print("=" * 60)
    t0 = time.time()
    try:
        ns = build_city(city)

        # Gate on the notebook's own validation, not just on "it didn't crash".
        checks = ns.get("CHECKS") or []
        failing = [name for name, ok, _ in checks if not ok]
        if not checks:
            raise RuntimeError(
                "no validation results found - did the notebook's validation section run?")
        if failing:
            raise RuntimeError(f"validation failed: {', '.join(failing)}")
        print(f"   {len(checks)} validation checks passed")

        published[city] = export_city(city)
        print(f"--> done in {time.time() - t0:.0f}s\n")
    except Exception as exc:
        failed[city] = str(exc)
        print(f"--> FAILED: {exc}")
        traceback.print_exc()
        print()

print(f"\nTotal: {time.time() - run_start:.0f}s")
print(f"Published {len(published)} of {len(CITIES)} cities")
for city, path in published.items():
    print(f"   {city:<30} {path}")
if failed:
    print("\nFAILED:")
    for city, err in failed.items():
        print(f"   {city:<30} {err}")

## Verify the snapshot is loadable

Publishing something nobody checked is how you find out at 9am on event day. This reads each
exported folder back and reports row counts, using the same wildcard `load.sh` will use.

In [ ]:
ok = True
for city, base in published.items():
    print(f"\n{city}")
    for table in TABLES:
        uri = f"{base}/{table}/*.parquet"
        try:
            # Read straight from GCS via an external table definition,
            # using the same wildcard load.sh will use.
            cfg = bigquery.ExternalConfig("PARQUET")
            cfg.source_uris = [uri]
            job_cfg = bigquery.QueryJobConfig(
                table_definitions={"snap": cfg})
            rows = client.query(
                "SELECT COUNT(*) AS n FROM snap", job_config=job_cfg
            ).to_dataframe().iloc[0]["n"]
            flag = "" if rows > 0 else "   <-- EMPTY"
            print(f"   {table:<22} {int(rows):>10,} rows{flag}")
            if rows == 0:
                ok = False
        except Exception as exc:
            print(f"   {table:<22} UNREADABLE: {exc}")
            ok = False

print("\n" + ("Snapshot verified. data_plane/scripts/load.sh will work."
              if ok else "PROBLEMS FOUND - do not rely on this snapshot."))